In [1]:
# ==========================================
# СБОР МЕТАДАННЫХ ИЗ ВСЕХ PDF В ПАПКЕ
# С КРАТКИМ СОДЕРЖАНИЕМ НА РУССКОМ ЯЗЫКЕ
# ==========================================
import re
import json
import time
import requests
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd

In [8]:
# ==========================================
# НАСТРОЙКИ
# ==========================================
PDF_FOLDER = Path(r"G:\Мой диск\Стажировка\BESS\add")   # <<< УКАЖИ СВОЮ ПАПКУ

OUTPUT_JSON = PDF_FOLDER / "pdf_metadata.json"
OUTPUT_CSV = PDF_FOLDER / "pdf_metadata.csv"

MAX_TEXT_PAGES = 10
MAX_CHARS_FOR_ANALYSIS = 15000

# ---- OLLAMA / LLM ----
OLLAMA_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "gpt-oss:120b-cloud"   # можешь заменить при необходимости

REQUEST_TIMEOUT = 180
SLEEP_BETWEEN_FILES = 0.3

In [10]:
# ==========================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==========================================

def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()


def extract_text_from_first_pages(pdf_path: Path, max_pages: int = 10) -> str:
    text_parts = []

    with fitz.open(pdf_path) as doc:
        pages_to_read = min(len(doc), max_pages)

        for i in range(pages_to_read):
            try:
                page_text = doc[i].get_text("text")
                if page_text and page_text.strip():
                    text_parts.append(page_text)
            except Exception:
                pass

    return clean_text("\n".join(text_parts))


def detect_language(text: str) -> str:
    if not text:
        return "unknown"

    sample = text[:5000]

    cyrillic = len(re.findall(r"[А-Яа-яЁё]", sample))
    latin = len(re.findall(r"[A-Za-z]", sample))

    if cyrillic > latin * 1.5:
        return "ru"
    elif latin > cyrillic * 1.5:
        return "en"
    elif cyrillic > 0 and latin > 0:
        return "mixed"
    else:
        return "unknown"


def extract_year(pdf_meta: dict, text: str, filename: str):
    # 1. из встроенных PDF metadata
    for key in ["creationDate", "modDate"]:
        value = pdf_meta.get(key)
        if value:
            years = re.findall(r"(19\d{2}|20\d{2})", str(value))
            if years:
                return years[0]

    # 2. из первых страниц
    years = re.findall(r"\b(19\d{2}|20\d{2})\b", text[:5000])
    if years:
        years_sorted = sorted(set(years))
        return years_sorted[0]

    # 3. из имени файла
    years = re.findall(r"\b(19\d{2}|20\d{2})\b", filename)
    if years:
        return years[0]

    return None


def extract_author(pdf_meta: dict, text: str):
    # 1. из metadata
    author = pdf_meta.get("author")
    if author and author.strip():
        return author.strip()

    # 2. пробуем найти на первых строках
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    first_lines = lines[:20]

    patterns = [
        r"^(?:author|authors|автор|авторы)\s*[:\-]\s*(.+)$",
    ]

    for line in first_lines:
        for pattern in patterns:
            m = re.match(pattern, line, flags=re.IGNORECASE)
            if m:
                value = m.group(1).strip()
                if 2 < len(value) < 200:
                    return value

    # 3. иногда автор идёт второй строкой после заголовка
    if len(first_lines) >= 2:
        candidate = first_lines[1]
        if 3 <= len(candidate) <= 120:
            if len(candidate.split()) <= 8 and not re.search(r"\b(19\d{2}|20\d{2})\b", candidate):
                return candidate

    return None


def extract_title(pdf_meta: dict, text: str, filename: str) -> str:
    # 1. из metadata
    title = pdf_meta.get("title")
    if title and title.strip() and title.strip().lower() not in {"untitled", ""}:
        return title.strip()

    # 2. из первых строк текста
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in lines[:15]:
        if len(line) >= 5 and len(line) < 250:
            if not line.lower().startswith("http"):
                return line

    # 3. из имени файла
    return Path(filename).stem


def extract_page_count(pdf_path: Path) -> int:
    with fitz.open(pdf_path) as doc:
        return len(doc)


def build_short_summary_fallback(text: str) -> str:
    if not text:
        return "Не удалось извлечь текст для краткого содержания."

    text = re.sub(r"\s+", " ", text).strip()
    text = text[:1500]

    sentences = re.split(r"(?<=[\.\!\?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 40]

    if not sentences:
        return text[:700]

    summary = " ".join(sentences[:4]).strip()

    if len(summary) > 1200:
        summary = summary[:1200].rsplit(" ", 1)[0] + "..."

    return summary


def call_ollama_generate(prompt: str, model: str = LLM_MODEL) -> str:
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        },
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    data = response.json()
    return data.get("response", "").strip()


def generate_summary_ru(text: str) -> str:
    if not text.strip():
        return "Не удалось извлечь текст для краткого содержания."

    text = text[:MAX_CHARS_FOR_ANALYSIS]

    prompt = f"""
Ты анализируешь текст книги или учебного материала.

Задача:
1. Определи основную тему текста.
2. Составь краткое содержание строго на русском языке.
3. Если исходный текст не на русском, передай смысл на русском.
4. Пиши кратко, ясно, без воды.
5. Объём: 3-5 предложений.
6. Не добавляй ничего, чего нет в тексте.

Текст:
{text}

Краткое содержание на русском:
""".strip()

    try:
        result = call_ollama_generate(prompt)
        if result:
            return result
        return build_short_summary_fallback(text)
    except Exception as e:
        return f"{build_short_summary_fallback(text)} [LLM summary error: {e}]"


def extract_pdf_metadata(pdf_path: Path) -> dict:
    try:
        with fitz.open(pdf_path) as doc:
            pdf_meta = doc.metadata or {}
            page_count = len(doc)

        text = extract_text_from_first_pages(pdf_path, max_pages=MAX_TEXT_PAGES)
        language = detect_language(text)

        filename = pdf_path.name
        title = extract_title(pdf_meta, text, filename)
        author = extract_author(pdf_meta, text)
        year = extract_year(pdf_meta, text, filename)
        summary_ru = generate_summary_ru(text)

        return {
            "title": title,
            "author": author,
            "publication_year": year,
            "language": language,
            "short_summary": summary_ru,
            "pages": page_count,
            "file_name": filename
        }

    except Exception as e:
        return {
            "title": pdf_path.stem,
            "author": None,
            "publication_year": None,
            "language": "unknown",
            "short_summary": f"Ошибка обработки файла: {e}",
            "pages": None,
            "file_name": pdf_path.name
        }


# ==========================================
# ОСНОВНАЯ ФУНКЦИЯ
# ==========================================

def collect_pdf_metadata_from_folder(folder_path: Path):
    pdf_files = sorted(folder_path.glob("*.pdf"))

    if not pdf_files:
        print(f"PDF-файлы в папке не найдены: {folder_path}")
        return []

    results = []

    print(f"Найдено PDF-файлов: {len(pdf_files)}\n")

    for i, pdf_file in enumerate(pdf_files, start=1):
        print("=" * 70)
        print(f"[{i}/{len(pdf_files)}] Обработка: {pdf_file.name}")

        metadata = extract_pdf_metadata(pdf_file)
        results.append(metadata)

        print("Название:", metadata["title"])
        print("Автор:", metadata["author"])
        print("Год:", metadata["publication_year"])
        print("Язык:", metadata["language"])
        print("Страниц:", metadata["pages"])
        print("Файл:", metadata["file_name"])
        print("Краткое содержание:", metadata["short_summary"][:300], "...\n")

        time.sleep(SLEEP_BETWEEN_FILES)

    return results

In [14]:
# ==========================================
# ЗАПУСК
# ==========================================

results = collect_pdf_metadata_from_folder(PDF_FOLDER)

# Сохранение в JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Сохранение в CSV
# df = pd.DataFrame(results)
# df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# # print("\n" + "=" * 70)
# print("ГОТОВО")
# print("JSON:", OUTPUT_JSON)
# print("CSV :", OUTPUT_CSV)
# print(f"Всего обработано файлов: {len(results)}")

# Красивый вывод JSON в консоль
# print("\nПРИМЕР JSON:")
# print(json.dumps(results[:3], ensure_ascii=False, indent=2))

Найдено PDF-файлов: 5

[1/5] Обработка: 199652305.pdf
Название: Весці Нацыянальнай акадэміі навук Беларусі. Серыя хімічных навук. 2024. Т. 60, № 2. C. 115–120 115
Автор: ISSN 1561-8331 (Print)
Год: 2024
Язык: ru
Страниц: 6
Файл: 199652305.pdf
Краткое содержание: Основная тема текста — исследование положительного электрода из цинк‑марганцевой шпинели (ZnMn₂O₄) для неводных цинк‑ионных аккумуляторов. Путём частичного растворения шпинели в растворе H₂SO₄ удалось увеличить её удельную поверхность в десятки раз, что было подтверждено адсорбцией N₂. Увеличение пл ...

[2/5] Обработка: 28-36.pdf
Название: Журнал Белорусского государственного университета. Химия. 2023;1:28–36
Автор: None
Год: 2023
Язык: mixed
Страниц: 9
Файл: 28-36.pdf
Краткое содержание: Основная тема текста — исследование сверхрешёток Bi₅Te₃ как нового катодного материала для водных цинк‑ионных аккумуляторов. Был разработан прототип аккумулятора, в котором в процессе разряда происходит подпотенциальное осаждение Zn на катодн